# Detección de emociones en tiempo real con Roboflow + Google Colab

Este notebook consume el modelo de Roboflow Universe:

`human-face-emotions/28`

Flujo:
1. Instala dependencias.
2. Configura tu API Key de Roboflow.
3. Prueba el modelo con una imagen.
4. Activa la webcam en Colab usando JavaScript.
5. Envía frames al endpoint Serverless de Roboflow y dibuja las predicciones.

> Nota: en Colab, `cv2.VideoCapture(0)` normalmente no puede acceder a tu webcam local porque el runtime corre en una VM remota. Por eso se usa JavaScript para capturar frames desde el navegador.


In [1]:
#@title 1) Instalar dependencias
!pip -q install inference-sdk opencv-python-headless pillow numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 860.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
#@title 2) Imports
import os
import time
import json
import getpass
from base64 import b64decode

import cv2
import numpy as np
from PIL import Image as PILImage

from IPython.display import display, Javascript, Image, Markdown
from google.colab.output import eval_js
from google.colab import files

from inference_sdk import InferenceHTTPClient


ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

In [ ]:
#@title 3) Configurar API Key y cliente Roboflow

# Opción recomendada:
# En Colab, ve a "Secrets" (icono de llave) y crea un secreto llamado ROBOFLOW_API_KEY.
# Si no existe, este notebook te pedirá pegar la clave manualmente.

MODEL_ID = "human-face-emotions/28"
API_URL = "https://serverless.roboflow.com"

try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = getpass.getpass("Pega tu API Key de Roboflow: ")

CLIENT = InferenceHTTPClient(
    api_url=API_URL,
    api_key=API_KEY
)

print(f"Cliente listo. Modelo: {MODEL_ID}")


In [ ]:
#@title 4) Funciones para dibujar predicciones

def _safe_predictions(result):
    """Devuelve una lista de predicciones aunque el formato cambie ligeramente."""
    if not isinstance(result, dict):
        return []

    preds = result.get("predictions", [])

    # Caso usual en object detection: lista de dicts
    if isinstance(preds, list):
        return preds

    # Algunos endpoints pueden devolver dicts anidados
    if isinstance(preds, dict):
        possible = []
        for value in preds.values():
            if isinstance(value, list):
                possible.extend(value)
        return possible

    return []


def draw_predictions(image_bgr, result, conf_threshold=0.35):
    """Dibuja bounding boxes y etiquetas sobre una imagen BGR de OpenCV."""
    preds = _safe_predictions(result)

    for pred in preds:
        try:
            confidence = float(pred.get("confidence", 0.0))
        except Exception:
            confidence = 0.0

        if confidence < conf_threshold:
            continue

        cls = str(pred.get("class", pred.get("class_name", "emotion")))

        # Roboflow object detection suele retornar centro x/y + width/height.
        if all(k in pred for k in ["x", "y", "width", "height"]):
            x = float(pred["x"])
            y = float(pred["y"])
            w = float(pred["width"])
            h = float(pred["height"])

            x1 = int(max(0, x - w / 2))
            y1 = int(max(0, y - h / 2))
            x2 = int(min(image_bgr.shape[1] - 1, x + w / 2))
            y2 = int(min(image_bgr.shape[0] - 1, y + h / 2))
        # Fallback por si viene en formato x_min/y_min/x_max/y_max.
        elif all(k in pred for k in ["x_min", "y_min", "x_max", "y_max"]):
            x1 = int(pred["x_min"])
            y1 = int(pred["y_min"])
            x2 = int(pred["x_max"])
            y2 = int(pred["y_max"])
        else:
            continue

        label = f"{cls} {confidence:.0%}"

        cv2.rectangle(image_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.rectangle(image_bgr, (x1, max(0, y1 - 24)), (x1 + 9 * len(label), y1), (0, 255, 0), -1)
        cv2.putText(
            image_bgr,
            label,
            (x1 + 4, max(16, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 0),
            2,
            cv2.LINE_AA,
        )

    return image_bgr


def show_bgr(image_bgr):
    """Muestra una imagen BGR en el notebook como JPEG."""
    ok, buffer = cv2.imencode(".jpg", image_bgr)
    if not ok:
        raise RuntimeError("No se pudo codificar la imagen.")
    display(Image(data=buffer.tobytes()))


def print_predictions(result, conf_threshold=0.35):
    preds = _safe_predictions(result)
    filtered = [
        p for p in preds
        if float(p.get("confidence", 0.0)) >= conf_threshold
    ]

    if not filtered:
        print("No se encontraron predicciones por encima del umbral.")
        return

    for i, p in enumerate(filtered, start=1):
        cls = p.get("class", p.get("class_name", "emotion"))
        conf = float(p.get("confidence", 0.0))
        print(f"{i}. {cls}: {conf:.2%}")


In [ ]:
#@title 5) Prueba rápida con una imagen subida

uploaded = files.upload()

if not uploaded:
    raise ValueError("No subiste ninguna imagen.")

image_path = next(iter(uploaded.keys()))
print("Imagen:", image_path)

result = CLIENT.infer(image_path, model_id=MODEL_ID)

print("\nRespuesta cruda:")
print(json.dumps(result, indent=2)[:3000])

print("\nPredicciones filtradas:")
print_predictions(result, conf_threshold=0.35)

image_bgr = cv2.imread(image_path)
annotated = draw_predictions(image_bgr.copy(), result, conf_threshold=0.35)

print("\nImagen anotada:")
show_bgr(annotated)


## Webcam en Colab

Ejecuta las siguientes celdas para activar la cámara del navegador. El loop enviará frames a Roboflow y mostrará la imagen anotada.

Recomendaciones:
- Empieza con `duration_seconds=30`.
- Sube `conf_threshold` si aparecen muchas detecciones falsas.
- Baja `quality` o aumenta `sleep_seconds` para reducir consumo de API y mejorar estabilidad.


In [ ]:
#@title 6) Helpers de webcam para Colab

def start_webcam(width=640, height=480):
    """Inicia la webcam del navegador y guarda el stream en variables JavaScript globales."""
    js = Javascript(f'''
    async function startEmotionCamera() {{
      if (window._emotionStream) {{
        window._emotionStream.getTracks().forEach(track => track.stop());
      }}

      const oldContainer = document.getElementById('emotion-webcam-container');
      if (oldContainer) oldContainer.remove();

      const container = document.createElement('div');
      container.id = 'emotion-webcam-container';
      container.style.margin = '8px 0';

      const title = document.createElement('div');
      title.textContent = 'Webcam activa para captura de frames';
      title.style.fontWeight = '600';
      title.style.marginBottom = '6px';

      const video = document.createElement('video');
      video.id = 'emotion-webcam-video';
      video.style.width = '{width}px';
      video.style.maxWidth = '100%';
      video.style.borderRadius = '8px';
      video.style.border = '1px solid #ddd';
      video.setAttribute('playsinline', '');

      const stream = await navigator.mediaDevices.getUserMedia({{
        video: {{ width: {width}, height: {height} }},
        audio: false
      }});

      container.appendChild(title);
      container.appendChild(video);
      document.body.appendChild(container);

      video.srcObject = stream;
      await video.play();

      window._emotionStream = stream;
      window._emotionVideo = video;

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      return true;
    }}
    ''')
    display(js)
    eval_js("startEmotionCamera()")
    print("Webcam iniciada. Acepta el permiso del navegador si aparece.")


def capture_frame(quality=0.70):
    """Captura un frame de la webcam activa y lo devuelve como imagen BGR de OpenCV."""
    data_url = eval_js(f'''
    (() => {{
      const video = window._emotionVideo;
      if (!video) {{
        throw new Error("La webcam no está iniciada. Ejecuta start_webcam() primero.");
      }}

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      const ctx = canvas.getContext('2d');
      ctx.drawImage(video, 0, 0);

      return canvas.toDataURL('image/jpeg', {quality});
    }})()
    ''')

    binary = b64decode(data_url.split(",")[1])
    array = np.frombuffer(binary, dtype=np.uint8)
    frame_bgr = cv2.imdecode(array, cv2.IMREAD_COLOR)

    if frame_bgr is None:
        raise RuntimeError("No se pudo decodificar el frame de la webcam.")

    return frame_bgr


def stop_webcam():
    """Detiene la webcam del navegador."""
    js = Javascript('''
    (() => {
      if (window._emotionStream) {
        window._emotionStream.getTracks().forEach(track => track.stop());
        window._emotionStream = null;
      }

      const container = document.getElementById('emotion-webcam-container');
      if (container) container.remove();

      window._emotionVideo = null;
      return true;
    })()
    ''')
    display(js)
    print("Webcam detenida.")


In [ ]:
#@title 7) Loop de detección de emociones en tiempo real

def run_realtime_emotion_detection(
    duration_seconds=30,
    conf_threshold=0.35,
    quality=0.65,
    sleep_seconds=0.10,
    width=640,
    height=480,
):
    """
    Ejecuta detección de emociones usando frames de la webcam.

    Parámetros:
    - duration_seconds: duración total del loop.
    - conf_threshold: confianza mínima para mostrar predicciones.
    - quality: calidad JPEG del frame enviado a Roboflow; menor = más rápido/menos datos.
    - sleep_seconds: pausa entre llamadas; mayor = menos consumo de API.
    - width, height: resolución solicitada al navegador.
    """
    start_webcam(width=width, height=height)

    display_handle = display(Markdown("Iniciando detección..."), display_id=True)

    frame_count = 0
    start_time = time.time()

    try:
        while True:
            elapsed = time.time() - start_time
            if elapsed >= duration_seconds:
                break

            frame_bgr = capture_frame(quality=quality)

            # El SDK acepta NumPy arrays. Convertimos a RGB para evitar ambigüedad de color.
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            result = CLIENT.infer(frame_rgb, model_id=MODEL_ID)

            annotated = draw_predictions(
                frame_bgr.copy(),
                result,
                conf_threshold=conf_threshold
            )

            frame_count += 1
            fps_avg = frame_count / max(time.time() - start_time, 1e-6)

            status = f"Frames: {frame_count} | FPS promedio: {fps_avg:.2f} | Umbral: {conf_threshold:.2f}"
            cv2.putText(
                annotated,
                status,
                (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

            ok, buffer = cv2.imencode(".jpg", annotated)
            if not ok:
                raise RuntimeError("No se pudo codificar el frame anotado.")

            display_handle.update(Image(data=buffer.tobytes()))

            if sleep_seconds > 0:
                time.sleep(sleep_seconds)

    except KeyboardInterrupt:
        print("Loop detenido manualmente.")
    finally:
        stop_webcam()
        print(f"Finalizado. Frames procesados: {frame_count}")


# Ejecuta esta línea para probar:
run_realtime_emotion_detection(
    duration_seconds=30,
    conf_threshold=0.35,
    quality=0.65,
    sleep_seconds=0.10,
    width=640,
    height=480,
)


## Ajustes útiles

Prueba estas variantes si necesitas optimizar:

```python
# Menos llamadas a la API
run_realtime_emotion_detection(duration_seconds=60, sleep_seconds=0.30, quality=0.55)

# Más sensibilidad
run_realtime_emotion_detection(duration_seconds=30, conf_threshold=0.20)

# Menos falsos positivos
run_realtime_emotion_detection(duration_seconds=30, conf_threshold=0.50)
```

## Solución de problemas

- **`Permission denied` o no aparece cámara**: permite el acceso a la cámara en el navegador y vuelve a ejecutar la celda.
- **Va lento**: baja `quality`, baja `width/height` o sube `sleep_seconds`.
- **Mucho consumo de API**: usa una duración menor y aumenta `sleep_seconds`.
- **No dibuja cajas**: revisa la respuesta cruda del modelo en la prueba con imagen; si el formato de predicción cambia, ajusta `draw_predictions()`.
